# GPT Action 库：Notion

## 简介

本页面为开发者针对特定应用构建 GPT Action 提供说明与指南。在继续之前，请先熟悉以下内容： 
- [GPT Actions 简介](https://platform.openai.com/docs/actions)
- [GPT Actions 库简介](https://platform.openai.com/docs/actions/actions-library)
- [从零构建 GPT Action 的示例](https://platform.openai.com/docs/actions/getting-started)

这个 GPT Action 概述了如何连接 **Notion**。该 Action 接收用户的问题，利用 Notion 的搜索功能扫描相关的 Notion 页面，然后返回匹配页面的信息。

### 价值与业务用例示例

**价值**：用户现在可以利用 ChatGPT 的自然语言能力，直接连接、查询并综合其在 Notion 中的知识。管理员可以显式地将页面共享给该集成，以此管理访问权限。

**用例示例**： 
- 新员工希望快速获取配置新系统的操作说明
- 支持人员需要快速从 Notion 中检索信息，而无需通读整篇文档
- 用户希望综合信息，生成摘要或转换后的内容，用于工作的其他方面

## 应用信息

### 应用关键链接

开始之前，请先查看该应用的以下链接：
- 应用网站：https://www.notion.so/
- 应用 API 文档：https://developers.notion.com/reference/intro
- Notion 授权方式：https://developers.notion.com/docs/authorization
    - 注意：Notion 仅允许“Public Integrations”使用 OAuth。请参阅链接中的文档，确定哪种方式最适合你的需求

### 应用前提条件

开始之前，请确保在你的应用环境中完成以下步骤：
- 设置一个包含已填充页面的 Notion 工作区
- 在 Notion 中共享页面时，针对特定 Wiki 效果最好。可以考虑将知识库组织成一个或一组 Wiki

## ChatGPT 步骤

### 自定义 GPT 指令 

创建自定义 GPT 后，将下面的文本复制到 Instructions 面板中。有疑问？请查看 [入门示例](https://cookbook.openai.com/examples/chatgpt/gpt_actions_library/.gpt_action_getting_started)，详细了解这一步的工作方式。

In [ ]:
**Context**: You are a helpful chatbot focussed on retrieving information from a company's Notion. An administrator has given you access to a number of useful Notion pages.  You are to act similar to a librarian and be helpful answering and finding answers for users' questions.

**Instructions**:
1. Use the search functionality to find the most relevant page or pages.
- Display the top 3 pages.  Include a formatted list containing: Title, Last Edit Date, Author.
- The Title should be a link to that page.
1.a. If there are no relevant pages, reword the search and try again (up to 3x)
1.b. If there are no relevant pages after retries, return "I'm sorry, I cannot find the right info to help you with that question"
2. Open the most relevant article, retrieve and read all of the contents (including any relevant linked pages or databases), and provide a 3 sentence summary.  Always provide a quick summary before moving to the next step.
3. Ask the user if they'd like to see more detail.  If yes, provide it and offer to explore more relevant pages.

**Additional Notes**: 
- If the user says "Let's get started", introduce yourself as a librarian for the Notion workspace, explain that the user can provide a topic or question, and that you will help to look for relevant pages.
- If there is a database on the page.  Always read the database when looking at page contents.

### OpenAPI Schema 

创建自定义 GPT 后，将下面的文本复制到 Actions 面板中。有疑问？请查看 [入门示例](https://platform.openai.com/docs/actions/getting-started)，详细了解这一步的工作方式。

In [ ]:
openapi: 3.1.0
info:
  title: Notion API
  description: API for interacting with Notion's pages, databases, and users.
  version: 1.0.0
servers:
  - url: https://api.notion.com/v1
    description: Main Notion API server
paths:
  /users:
    get:
      operationId: listAllUsers
      summary: List all users
      parameters:
        - name: Notion-Version
          in: header
          required: true
          schema:
            type: string
          example: 2022-06-28
          constant: 2022-06-28
      responses:
        '200':
          description: Successful response
          content:
            application/json:
              schema:
                type: object
                properties:
                  results:
                    type: array
                    items:
                      type: object
                      properties:
                        id:
                          type: string
                        name:
                          type: string
                        avatar_url:
                          type: string
                        type:
                          type: string
  /blocks/{block_id}/children:
    get:
      operationId: retrieveBlockChildren
      summary: Retrieve block children
      parameters:
        - name: block_id
          in: path
          required: true
          schema:
            type: string
        - name: Notion-Version
          in: header
          required: true
          schema:
            type: string
          example: 2022-06-28
          constant: 2022-06-28
      responses:
        '200':
          description: Successful response
          content:
            application/json:
              schema:
                type: object
                properties:
                  object:
                    type: string
                  results:
                    type: array
                    items:
                      type: object
                      properties:
                        id:
                          type: string
                        type:
                          type: string
                        has_children:
                          type: boolean
  /comments:
    get:
      operationId: retrieveComments
      summary: Retrieve comments
      parameters:
        - name: Notion-Version
          in: header
          required: true
          schema:
            type: string
          example: 2022-06-28
          constant: 2022-06-28
      responses:
        '200':
          description: Successful response
          content:
            application/json:
              schema:
                type: object
                properties:
                  results:
                    type: array
                    items:
                      type: object
                      properties:
                        id:
                          type: string
                        text:
                          type: string
                        created_time:
                          type: string
                          format: date-time
                        created_by:
                          type: object
                          properties:
                            id:
                              type: string
                            name:
                              type: string
  /pages/{page_id}/properties/{property_id}:
    get:
      operationId: retrievePagePropertyItem
      summary: Retrieve a page property item
      parameters:
        - name: page_id
          in: path
          required: true
          schema:
            type: string
        - name: property_id
          in: path
          required: true
          schema:
            type: string
        - name: Notion-Version
          in: header
          required: true
          schema:
            type: string
          example: 2022-06-28
          constant: 2022-06-28
      responses:
        '200':
          description: Successful response
          content:
            application/json:
              schema:
                type: object
                properties:
                  id:
                    type: string
                  type:
                    type: string
                  title:
                    type: array
                    items:
                      type: object
                      properties:
                        type:
                          type: string
                        text:
                          type: object
                          properties:
                            content:
                              type: string
  /databases/{database_id}/query:
    post:
      operationId: queryDatabase
      summary: Query a database
      parameters:
        - name: database_id
          in: path
          required: true
          schema:
            type: string
        - name: Notion-Version
          in: header
          required: true
          schema:
            type: string
          example: 2022-06-28
          constant: 2022-06-28
      requestBody:
        required: true
        content:
          application/json:
            schema:
              type: object
              properties:
                filter:
                  type: object
                sorts:
                  type: array
                  items:
                    type: object
                start_cursor:
                  type: string
                page_size:
                  type: integer
      responses:
        '200':
          description: Successful response
          content:
            application/json:
              schema:
                type: object
                properties:
                  object:
                    type: string
                  results:
                    type: array
                    items:
                      type: object
                  next_cursor:
                    type: string
                  has_more:
                    type: boolean
  /search:
    post:
      operationId: search
      summary: Search
      parameters:
        - name: Notion-Version
          in: header
          required: true
          schema:
            type: string
          example: 2022-06-28
          constant: 2022-06-28
      requestBody:
        required: true
        content:
          application/json:
            schema:
              type: object
              properties:
                query:
                  type: string
                filter:
                  type: object
                  properties:
                    value:
                      type: string
                    property:
                      type: string
                sort:
                  type: object
                  properties:
                    direction:
                      type: string
                    timestamp:
                      type: string
      responses:
        '200':
          description: Successful response
          content:
            application/json:
              schema:
                type: object
                properties:
                  object:
                    type: string
                  results:
                    type: array
                    items:
                      type: object
                      properties:
                        id:
                          type: string
                        title:
                          type: array
                          items:
                            type: object
                            properties:
                              type:
                                type: string
                              text:
                                type: object
                                properties:
                                  content:
                                    type: string

## 身份验证说明

以下是为该第三方应用设置身份验证的说明。

### Action 前置步骤

在 ChatGPT 中设置身份验证之前，请先在 Notion 中完成以下步骤。

1. 进入你工作区的 Notion 设置页面
2. 导航到 My Connections > Develop or Manage Integrations
3. 创建一个标记为 Internal 的新集成
4. 找到你的集成，并找到标记为 Internal Integration Secret 的 API Key。这就是该集成的 bearer token。

**注意！** 你需要将特定页面、数据库或 Wiki 共享给该集成，才能在 ChatGPT 中访问它们。方法是点击页面右上角的 ... 按钮，然后选择相应的连接。

**注意！** 如果集成被标记为“Public”，Notion 允许其使用 OAuth。请查阅 [Notion 身份验证文档](https://developers.notion.com/docs/authorization)，确定哪种集成方式最适合你的需求。

![notion_connections.png](../../../images/creating_notion_integration.png)

![sharing_notion_pages.png](../../../images/sharing_notion_with_GPT.png)

### 在 ChatGPT 中

在 ChatGPT 中，点击“Authentication”并选择 **“API Key”**。填写以下信息。 

- **API Key**：使用上述步骤中获得的 Internal Integration Secret
- **Auth Type**：Bearer

### 常见问题与故障排查

- *搜索没有返回任何结果* 如果运行搜索时没有返回任何页面，请再次确认你已在 Notion 中将相关页面共享给该应用

*有希望我们优先支持的集成吗？我们的集成中有错误吗？请在我们的 GitHub 中提交 PR 或 issue，我们会及时查看。*
